# C-01 Kaggle CPU environment smoke

Import this notebook into Kaggle and select **Accelerator → None**. Choose one source for the project files in the configuration cell.

1. **Automatic private GitHub checkout.** Leave `CHECKOUT` and `KAGGLE_DATASET_ROOT` unset. Enable **Internet** and add a Kaggle Secret named `GITHUB_TOKEN`. Grant the notebook access to that secret. Use a fine-grained GitHub token with **Contents: Read** permission for `AIVIETNAM-AIO-DucTam/SGBR-Reproduction`. Set `GITHUB_REF` to a branch name or a full 40-character commit SHA. A matching clean checkout is reused; set `GITHUB_REFRESH = True` to fetch the branch again.
2. **Private Kaggle Dataset.** Upload `scripts/kaggle_environment_smoke.py` from an authorized copy, preserving the `scripts/` folder. Attach the Dataset with **Add Input** and set `KAGGLE_DATASET_ROOT` to the directory immediately above `scripts/`, for example `Path("/kaggle/input/your-dataset/SGBR-Reproduction")`. Leave `CHECKOUT` unset.
3. **Existing local checkout.** Set `CHECKOUT` to a checkout already available to the notebook and leave `KAGGLE_DATASET_ROOT` unset.

Use the notebook and helper from the same project revision. The notebook checks the helper before loading it and writes output to `/kaggle/working`. The manifest records the resolved project commit as `project_git_commit` when Git metadata is available. A Dataset containing only source files has no commit metadata.

Fill in the session timeout and source shown for your notebook. Set the execution mode before selecting **Save Version → Run All** for a batch run. For an online build, enable Internet so the notebook can fetch the pinned toolchain and upstream MOA revision. For offline loading, attach the verified JDK archive (or provide an installed JDK) and a trusted MOA fat jar with its recorded build JDK and SHA-256. The staged jar must come from the pinned authors' fork.

The default is Temurin JDK 11. The pinned MOA build failed on JDK 8 because MEKA 1.9.7 contains Java 11 bytecode. JVM choice can affect timing.

The eight synthetic rows exercise CLI wiring and learner construction. C-02 handles Abalone parity separately. The output is `/kaggle/working/results/manifest/environment.json`.

In [ ]:
from pathlib import Path

# Leave both paths unset to use automatic private GitHub checkout.
KAGGLE_DATASET_ROOT = None  # Example: Path("/kaggle/input/your-dataset/SGBR-Reproduction")
CHECKOUT = None  # Example: Path("/kaggle/working/SGBR-Reproduction")
GITHUB_REF = "main"  # Branch name or full 40-character commit SHA.
GITHUB_REFRESH = False  # Fetch again instead of reusing a matching checkout.
SESSION_LIMIT_HOURS = None  # Fill in the limit for this Kaggle CPU session.
SESSION_LIMIT_SOURCE = ""  # Cite the Kaggle UI or applicable documentation.
EXECUTION_MODE = "interactive"  # Use "batch" for Save Version → Run All.
KAGGLE_IMAGE = None  # Record the image/version shown by Kaggle, if available.

# Optional: use a staged JDK or a documented alternate JDK.
JAVA_HOME_OVERRIDE = None
# Update this note if you choose a different JVM through JAVA_HOME_OVERRIDE.
JAVA_SELECTION_NOTE = "The pinned MOA build depends on MEKA 1.9.7, which requires Java 11 bytecode support."
# Optional: staged tool archives with the same hashes as the online downloads.
JDK_ARCHIVE = None
MAVEN_ARCHIVE = None
# Optional: load the pinned fork's jar from a trusted Kaggle Dataset.
# Supply its recorded SHA-256 and build JDK together with the path.
MOA_JAR = None
MOA_SHA256 = ""
MOA_BUILD_JDK = ""


In [ ]:
import os
import re
import subprocess
from tempfile import TemporaryDirectory

PROJECT_URL = "https://github.com/AIVIETNAM-AIO-DucTam/SGBR-Reproduction.git"
DEFAULT_CHECKOUT = Path("/kaggle/working/SGBR-Reproduction")


def project_git(folder, *args, env=None):
    try:
        return subprocess.run(
            ["git", "-C", str(folder), *args], capture_output=True, text=True,
            env=env, timeout=180, check=False,
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError("Git timed out. Enable Internet in Kaggle and check GitHub connectivity.") from None
    except OSError:
        raise RuntimeError("Could not start Git. Check that Git is available in the notebook.") from None


def project_commit(folder):
    if not (folder / ".git").exists():
        return None
    head = project_git(folder, "rev-parse", "--verify", "HEAD^{commit}")
    if head.returncode or not re.fullmatch(r"[0-9a-f]{40}", head.stdout.strip()):
        raise RuntimeError("Could not resolve the project checkout's Git commit.")
    return head.stdout.strip()


def fetch_project(folder, revision):
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError(
            "Cannot read Kaggle Secret GITHUB_TOKEN. Add the fine-grained token in "
            "Kaggle Secrets and grant this notebook access to it."
        ) from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is empty. Set it in Kaggle Secrets.")

    with TemporaryDirectory(prefix="sgbr-git-auth-") as temporary:
        askpass = Path(temporary) / "askpass"
        askpass.write_text(
            '#!/bin/sh\ncase "$1" in\n'
            '  "Username for \'https://github.com\': "*) printf "%s\\n" x-access-token ;;\n'
            '  "Password for \'https://x-access-token@github.com\': "*) '
            'printf "%s\\n" "$SGBR_GIT_SECRET" ;;\n'
            '  *) exit 1 ;;\nesac\n',
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        env = {
            key: value for key, value in os.environ.items()
            if not key.startswith("GIT_") and key != "SSLKEYLOGFILE"
        }
        env.update(
            LC_ALL="C", GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0",
            GIT_CONFIG_NOSYSTEM="1", GIT_CONFIG_GLOBAL=os.devnull, SGBR_GIT_SECRET=token,
        )
        fetch = project_git(
            folder, "-c", "credential.helper=", "-c", "http.followRedirects=false",
            "-c", "http.sslVerify=true", "fetch", "--quiet", "--depth=1", "--no-tags",
            "--", PROJECT_URL, revision, env=env,
        )
    if not fetch.returncode:
        return

    # Classify the captured diagnostic without replaying Git or server output.
    diagnostic = fetch.stderr.lower()
    if any(message in diagnostic for message in (
        "could not resolve", "failed to connect", "couldn't connect", "network is unreachable",
        "connection timed out", "connect tunnel failed", "proxy", "connection reset",
    )):
        raise RuntimeError("Could not reach GitHub. Enable Internet in Kaggle notebook settings and retry.")
    if any(message in diagnostic for message in (
        "authentication failed", "repository not found", "401", "403",
        "access denied", "could not read username", "could not read password", "invalid username or token",
    )):
        raise RuntimeError(
            "GitHub authentication or repository access failed. Check that GITHUB_TOKEN is valid "
            "and has Contents: Read access to AIVIETNAM-AIO-DucTam/SGBR-Reproduction."
        )
    if any(message in diagnostic for message in ("couldn't find remote ref", "not our ref", "unadvertised object")):
        raise RuntimeError("GitHub could not fetch GITHUB_REF. Check the branch name or full commit SHA.")
    raise RuntimeError("GitHub fetch failed. Verify Internet, token permissions, and GITHUB_REF, then retry.")


def prepare_project_checkout(target, ref, refresh=False):
    if not target.parent.is_dir():
        raise RuntimeError("Automatic checkout requires /kaggle/working. Run in Kaggle or set CHECKOUT.")
    if not isinstance(ref, str) or not ref or ref.startswith("-"):
        raise ValueError("Set GITHUB_REF to a branch name or full 40-character commit SHA.")
    is_sha = re.fullmatch(r"[0-9a-fA-F]{40}", ref) is not None
    revision = ref.lower() if is_sha else f"refs/heads/{ref}"
    if not is_sha and project_git(target.parent, "check-ref-format", revision).returncode:
        raise ValueError("Set GITHUB_REF to a valid branch name or full 40-character commit SHA.")

    if target.exists() or target.is_symlink():
        if target.is_symlink() or not (target / ".git").is_dir():
            raise RuntimeError("Automatic checkout path is occupied. Set CHECKOUT to use it explicitly.")
        origin = project_git(target, "config", "--local", "--get", "remote.origin.url")
        if origin.returncode or origin.stdout.strip() != PROJECT_URL:
            raise RuntimeError("Automatic checkout path has a different Git origin. Set CHECKOUT to use it explicitly.")
        status = project_git(target, "status", "--porcelain")
        if status.returncode or status.stdout.strip():
            raise RuntimeError("Automatic checkout has local changes. Preserve them or select an explicit CHECKOUT.")
        head = project_commit(target)
        cached_ref = f"refs/remotes/origin/{ref}"
        cached = head == revision if is_sha else (
            project_git(target, "rev-parse", "--verify", cached_ref).stdout.strip() == head
        )
        if cached and not refresh:
            return head
        return checkout_project_revision(target, ref, revision, is_sha)

    # Build a shallow checkout in a sibling directory and publish it only on success.
    with TemporaryDirectory(prefix=".sgbr-checkout-", dir=target.parent) as temporary:
        folder = Path(temporary) / "repo"
        folder.mkdir()
        initial = project_git(folder, "init", "--quiet")
        if initial.returncode or project_git(folder, "remote", "add", "origin", PROJECT_URL).returncode:
            raise RuntimeError("Could not initialize the private checkout in /kaggle/working.")
        head = checkout_project_revision(folder, ref, revision, is_sha)
        folder.rename(target)
        return head


def checkout_project_revision(folder, ref, revision, is_sha):
    fetch_project(folder, revision)
    fetched = project_git(folder, "rev-parse", "--verify", "FETCH_HEAD^{commit}")
    sha = fetched.stdout.strip()
    if fetched.returncode or not re.fullmatch(r"[0-9a-f]{40}", sha) or (is_sha and sha != revision):
        raise RuntimeError("GitHub did not return the requested project commit.")
    if project_git(folder, "-c", "core.hooksPath=/dev/null", "checkout", "--quiet", "--detach", sha).returncode:
        raise RuntimeError("Could not check out the fetched project commit.")
    if not is_sha and project_git(folder, "update-ref", f"refs/remotes/origin/{ref}", sha).returncode:
        raise RuntimeError("Could not record the fetched project branch.")
    return project_commit(folder)


In [ ]:
import importlib.util

if CHECKOUT is not None and KAGGLE_DATASET_ROOT is not None:
    raise ValueError("Set only one of KAGGLE_DATASET_ROOT and CHECKOUT in the configuration cell.")

source_root = CHECKOUT
if KAGGLE_DATASET_ROOT is not None:
    source_root = KAGGLE_DATASET_ROOT
resolved_project_commit = None
if source_root is None:
    source_root = DEFAULT_CHECKOUT
    resolved_project_commit = prepare_project_checkout(source_root, GITHUB_REF, GITHUB_REFRESH)

source_root = Path(source_root).expanduser().resolve()
probe_script = source_root / "scripts/kaggle_environment_smoke.py"
try:
    if not probe_script.is_file():
        raise FileNotFoundError("Expected a Python script at this path.")
    probe_source = probe_script.read_bytes()
except OSError as error:
    raise FileNotFoundError(
        f"C-01 setup could not read the required script at {probe_script}. "
        "Attach the private Kaggle Dataset and set KAGGLE_DATASET_ROOT, or set CHECKOUT "
        "to an existing local checkout. Point to the directory that contains scripts/ "
        "and preserve scripts/kaggle_environment_smoke.py."
    ) from error

spec = importlib.util.spec_from_file_location("kaggle_environment_smoke", probe_script)
probe_module = importlib.util.module_from_spec(spec)
# Load the verified path without creating __pycache__ in the project source.
exec(compile(probe_source, str(probe_script), "exec"), vars(probe_module))
if resolved_project_commit is None:
    resolved_project_commit = project_commit(source_root)
UPSTREAM = probe_module.UPSTREAM
UPSTREAM_COMMIT = probe_module.UPSTREAM_COMMIT
KAGGLE_WORKING = probe_module.KAGGLE_WORKING
run = probe_module.run


In [ ]:
import csv
import hashlib
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.request

JDK_NAME = "OpenJDK11U-jdk_x64_linux_hotspot_11.0.29_7.tar.gz"
JDK_URL = f"https://github.com/adoptium/temurin11-binaries/releases/download/jdk-11.0.29%2B7/{JDK_NAME}"
JDK_SHA256 = "3c8f2b53dd137cd86e54f40df96fd0fc56df72c749c06469e7eab216503bc7cf"
MAVEN_NAME = "apache-maven-3.9.11-bin.tar.gz"
MAVEN_URL = f"https://repo.maven.apache.org/maven2/org/apache/maven/apache-maven/3.9.11/{MAVEN_NAME}"
MAVEN_SHA512 = (
    "bcfe4fe305c962ace56ac7b5fc7a08b87d5abd8b7e89027ab251069faebee516b0"
    "ded8961445d6d91ec1985dfe30f8153268843c89aa392733d1a3ec956c9978"
)


def command(args, cwd, timeout=120):
    result = subprocess.run(
        args, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, timeout=timeout, check=False,
    )
    if result.returncode:
        raise RuntimeError(f"{args[0]} exited {result.returncode}\n{result.stdout[-5000:]}")
    return result.stdout


def digest(path, algorithm="sha256"):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, algorithm).hexdigest()


def unpack_tool(work, staged, url, expected, algorithm):
    archive = Path(staged) if staged else work / url.rsplit("/", 1)[1]
    if not staged:
        with urllib.request.urlopen(url, timeout=30) as response, archive.open("wb") as stream:
            shutil.copyfileobj(response, stream)
    if digest(archive, algorithm) != expected:
        raise RuntimeError(f"Checksum mismatch for {archive.name}")
    command(["tar", "-xzf", str(archive), "-C", str(work)], work)


def provision(work, staged_jar):
    if JAVA_HOME_OVERRIDE:
        jdk = Path(JAVA_HOME_OVERRIDE)
    elif platform.machine() == "x86_64":
        unpack_tool(work, JDK_ARCHIVE, JDK_URL, JDK_SHA256, "sha256")
        jdk = work / "jdk-11.0.29+7"
    else:
        raise RuntimeError("Set JAVA_HOME_OVERRIDE to a verified JDK for this architecture.")
    os.environ["JAVA_HOME"] = str(jdk)
    os.environ["PATH"] = f"{jdk / 'bin'}:{os.environ['PATH']}"
    # Verify the selected JVM explicitly. PATH alone could fall back to the image JVM.
    command([str(jdk / "bin/java"), "-version"], work)
    if not staged_jar:
        unpack_tool(work, MAVEN_ARCHIVE, MAVEN_URL, MAVEN_SHA512, "sha512")
        maven = work / "apache-maven-3.9.11/bin"
        os.environ["PATH"] = f"{maven}:{os.environ['PATH']}"


def resolve_jar(work, details):
    if MOA_JAR:
        if not re.fullmatch(r"[0-9a-fA-F]{64}", MOA_SHA256) or not MOA_BUILD_JDK.strip():
            raise ValueError("Provide the staged MOA jar's recorded SHA-256 and build JDK.")
        jar = Path(MOA_JAR).resolve(strict=True)
        if digest(jar) != MOA_SHA256.lower():
            raise RuntimeError("The staged MOA jar does not match its expected SHA-256.")
        details.update(source="staged", build_jdk=MOA_BUILD_JDK, pinned_commit_verified=None)
        return jar

    source = work / "source"
    source.mkdir()
    command(["git", "init", "-q"], source)
    command(["git", "fetch", "--depth=1", UPSTREAM, UPSTREAM_COMMIT], source, timeout=300)
    actual = command(["git", "rev-parse", "FETCH_HEAD"], source).strip()
    details.update(source="build", actual_commit=actual, pinned_commit_verified=actual == UPSTREAM_COMMIT)
    if actual != UPSTREAM_COMMIT:
        raise RuntimeError(f"Unexpected upstream revision {actual}")
    command(["git", "checkout", "--detach", actual], source, timeout=120)
    details["build_jdk"] = command(["java", "-version"], source).strip()
    build = command(
        ["mvn", "-B", "-ntp", "-C", "clean", "install", "-DskipTests=true",
         "-Dmaven.javadoc.skip=true", "-Dlatex.skipBuild=true"],
        source / "moa", timeout=1200,
    )
    print(build[-2500:])
    return source / "moa/target/moa-2023.04.1-SNAPSHOT-jar-with-dependencies.jar"


def smoke(work, jar, details):
    fixture = "@relation probe\n@attribute x numeric\n@attribute z numeric\n@attribute y numeric\n@data\n"
    fixture += "".join(f"{i},{i % 2},{i * 2}\n" for i in range(1, 9))
    (work / "probe.arff").write_text(fixture, encoding="utf-8")
    task = (
        "EvaluatePrequentialRegression -s (ArffFileStream -f probe.arff) "
        "-l (meta.StreamingGradientBoostedTrees -t -L 1.0 -m 75 -s 10 "
        "-l (meta.OzaBag -s 10 -l (trees.FIMTDD -s VarianceReductionSplitCriterion "
        "-g 50 -c 0.01 -e)) -Z 1) -i 8 -f 8 -d smoke.csv"
    )
    details.update(seed=1, expected_instances=8, heap="256m", task=task)
    command(["java", "-Xmx256m", "-cp", str(jar), "moa.DoTask", task], work, timeout=60)
    with (work / "smoke.csv").open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    if len(rows) != 1 or float(rows[0]["learning evaluation instances"]) != 8:
        raise RuntimeError(f"Expected one MOA CSV row with eight processed instances, got {rows}")
    details.update(passed=True, processed_instances=8)


In [ ]:
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError("Use a Kaggle CPU notebook with /kaggle/working available.")
if (not isinstance(SESSION_LIMIT_HOURS, (int, float)) or not math.isfinite(SESSION_LIMIT_HOURS)
        or SESSION_LIMIT_HOURS <= 0 or not SESSION_LIMIT_SOURCE.strip()):
    raise ValueError("Record the Kaggle CPU session limit and its source in the configuration cell.")
if EXECUTION_MODE not in {"interactive", "batch"}:
    raise ValueError("Set EXECUTION_MODE to interactive or batch.")

output = KAGGLE_WORKING / "results/manifest/environment.json"
initial_tools = {
    name: run([name, "-version"]) for name in ("java", "javac", "mvn")
}
print("Notebook Python", sys.version.split()[0])
for name, result in initial_tools.items():
    print(name, result["output"].splitlines()[:1])

moa = {"expected_commit": UPSTREAM_COMMIT}
synthetic = {"passed": False}
issues = []
stage = "toolchain provisioning"
original_environment = {key: os.environ.get(key) for key in ("PATH", "JAVA_HOME")}

with tempfile.TemporaryDirectory(prefix="sgbr-c01-", dir=KAGGLE_WORKING) as temp:
    work = Path(temp)
    probe_output = work / "probe.json"
    probe = [
        sys.executable, str(probe_script),
        "--output", str(probe_output), "--session-limit-hours", str(SESSION_LIMIT_HOURS),
        "--session-limit-source", SESSION_LIMIT_SOURCE,
        "--java-selection-note", JAVA_SELECTION_NOTE,
    ]
    if MOA_JAR:
        probe.append("--allow-missing-maven")
    try:
        provision(work, bool(MOA_JAR))
        stage = "environment probe"
        command(probe, work, timeout=180)
        stage = "MOA build or verification"
        jar = resolve_jar(work, moa)
        moa.update(sha256=digest(jar), size_bytes=jar.stat().st_size)
        stage = "Sgb(Oza) smoke"
        smoke(work, jar, synthetic)
    except (OSError, RuntimeError, ValueError, subprocess.SubprocessError) as error:
        issues.append(f"{stage}: {error}")
        raise
    finally:
        try:
            if not probe_output.exists():
                # Capture resources and tool availability even after provisioning fails.
                try:
                    subprocess.run(probe, cwd=work, timeout=180, check=False)
                except (OSError, subprocess.TimeoutExpired) as error:
                    issues.append(f"Could not collect the fallback environment probe: {error}")
            manifest = json.loads(probe_output.read_text()) if probe_output.exists() else {}
            if manifest.get("java_major") not in (None, 8):
                issues.append("A newer JVM was selected. Default GC and wall time can differ from JDK 8.")
            if manifest.get("tools", {}).get("mvn", {}).get("exit_code") != 0:
                issues.append("Maven is unavailable. A verified staged jar can still run on Java.")
            cgroup = manifest.get("cgroup", {})
            memory_cap = cgroup.get("memory_limit_bytes")
            if memory_cap is not None and memory_cap <= 48 * 1024**3:
                issues.append("The visible memory cap leaves no headroom for the E1 48 GiB heap.")
            if not cgroup.get("limits"):
                issues.append("The notebook did not expose cgroup limits. Host totals cannot establish quotas.")
            for name, result in manifest.get("network", {}).items():
                if result.get("http_status", result.get("exit_code")) not in (200, 0):
                    issues.append(f"Connectivity check failed for {name}.")
            image = KAGGLE_IMAGE or os.environ.get("KAGGLE_DOCKER_IMAGE") or os.environ.get("KAGGLE_IMAGE_VERSION")
            if not image:
                issues.append("Kaggle image version was unavailable in this session.")
            manifest.update(
                kaggle_image=image, execution_mode=EXECUTION_MODE, initial_tools=initial_tools,
                project_git_commit=resolved_project_commit,
                moa=moa, synthetic_smoke=synthetic, compatibility_issues=issues,
                notebook_smoke_passed=bool(
                    manifest.get("toolchain_smoke_passed") and synthetic["passed"]
                ),
            )
            output.parent.mkdir(parents=True, exist_ok=True)
            output.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
            for name in ("python", "java", "mvn"):
                tool = manifest.get("tools", {}).get(name, {})
                lines = tool.get("output", "").splitlines()
                print(f"Selected {name}:", lines[0] if lines else "unavailable")
            print("Manifest", output)
            print("Smoke passed", manifest["notebook_smoke_passed"])
            for issue in issues:
                print(issue)
        finally:
            for key, value in original_environment.items():
                if value is None:
                    os.environ.pop(key, None)
                else:
                    os.environ[key] = value
